# 02. clean

## 0. setup

In [1]:
import re

import gc
from pathlib import Path

import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

## 1. config

In [2]:
# cartel
in_ref = data_raw / 'cartel' / 'manual' / 'cartel_manual_v7.xlsx'
eea_membership_ref = 'infr_start'     # 'infr_start' | 'decision' : ref year for bloc membership

#  patents
in_pat = data_interim / 'pat_data.parquet'
in_ipc_isic = data_raw / 'external' / 'ipc4_to_isic_rev4_3.txt'

# outputs
out_treated   = data_proc / 'cartel_treated.parquet'    # infringement x nace3 x ctry
out_patpanel  = data_interim / 'pat_panel.parquet'      # ctry x isic3 x year
out_coarse    = data_interim / 'nace_coarse_review.xlsx'
out_infr      = data_proc / 'clean_infringements.xlsx'

print(f'cartel : {in_ref.name} -> {out_treated.name}')
print(f'patents: {in_pat.name} + {in_ipc_isic.name} -> {out_patpanel.name}')

cartel : cartel_manual_v7.xlsx -> cartel_treated.parquet
patents: pat_data.parquet + ipc4_to_isic_rev4_3.txt -> pat_panel.parquet


## 2. country scope

In [3]:
# bloc membership for cartel scope expansion
ref_ctry = pd.read_excel(in_ref, sheet_name='ref_ctry_timeline')
universe = set(ref_ctry['ctry_iso'])

def members(bloc, year):
    '''ISO2s that are members of `bloc` ('EEA'|'EU') in `year`.'''
    if pd.isna(year):
        year = 3000  # ref year unknown -> include everyone ever a member
    entry = f'{bloc.lower()}_entry_year'
    exit_ = f'{bloc.lower()}_exit_year'
    m = ref_ctry[(ref_ctry[entry] <= year) &
                 (ref_ctry[exit_].isna() | (year < ref_ctry[exit_]))]
    return set(m['ctry_iso'])

# patent country universe
europe_iso2 = {
    # EU27
    'AT','BE','BG','HR','CY','CZ','DK','EE','FI','FR','DE','GR','HU','IE',
    'IT','LV','LT','LU','MT','NL','PL','PT','RO','SK','SI','ES','SE',
    'IS','LI','NO','CH',                    # EFTA / EEA
    'GB',                                   # UK
    'AD','MC','SM','VA',                    # microstates
    'AL','BA','MK','ME','RS','XK',          # Western Balkans
    'BY','MD','UA','RU',                    # Eastern Europe
    'TR','GE','AM','AZ',                    # transcontinental / borderline
}

print(f'bloc universe: {len(universe)} | patent (europe): {len(europe_iso2)} | timeline rows {ref_ctry.shape[0]}')

bloc universe: 31 | patent (europe): 50 | timeline rows 31


## 3. cartel

### 3.1 filter infringements

In [4]:
cartels   = pd.read_excel(in_ref, sheet_name='cartels')
decisions = pd.read_excel(in_ref, sheet_name='decisions')
print('cartels', cartels.shape, '| decisions', decisions.shape)

# keep only usable, non-excluded infringements
infr = cartels[cartels['exclude_reason'].isna()].copy()
for col in ['decision_year', 'start_year', 'end_year']:
    infr[col] = pd.to_numeric(infr[col], errors='coerce').astype('Int64')

print(f'kept after exclude_reason: {len(infr)}')
print('decision-year span :', int(infr['decision_year'].min()), '-', int(infr['decision_year'].max()))
print('formation-year span:', int(infr['start_year'].min()),   '-', int(infr['start_year'].max()))
print('breakup-year span  :', int(infr['end_year'].min()),     '-', int(infr['end_year'].max()))

cartels (215, 23) | decisions (199, 10)
kept after exclude_reason: 164
decision-year span : 1982 - 2025
formation-year span: 1969 - 2018
breakup-year span  : 1976 - 2022


### 3.2 case-level fine
Amendments / re-adoptions were entered as additive deltas, so the per-case **sum** is the final fine (`min_count=1` keeps a genuine no-fine case as NaN, not 0).

In [5]:
decisions['fine_eur'] = pd.to_numeric(decisions['fine_eur'], errors='coerce')

fine_info = (decisions
                .groupby('case_id')['fine_eur']
                .sum(min_count=1)
                .rename('fine_eur')
                .reset_index())

has_amend = decisions.loc[
    decisions['decision_type'].isin(['amendment', 're-adoption']), 'case_id'].unique()

fine_info['has_amendment'] = fine_info['case_id'].isin(has_amend).astype(int)

infr = infr.merge(fine_info, on='case_id', how='left')

### 3.3 parse NACE and explode to (infringement × NACE4)

In [6]:
def split_nace(cell):
    if not isinstance(cell, str):
        return []
    out = []
    for tok in cell.split(','):
        digits = re.sub(r'[^0-9]', '', tok)
        if len(digits) >= 2:
            out.append(digits)
    return out

infr['seg_id'] = (infr['infringement_id'].astype(str) + '_'
                  + infr['segment_seq'].astype('Int64').astype(str))

infr['nace_list'] = infr['nace_code'].apply(split_nace)

nace_long = (infr.explode('nace_list')
             .rename(columns={'nace_list': 'nace'})
             .dropna(subset=['nace']))
nace_long['nace_level'] = nace_long['nace'].str.len()

# analysis grain is 3-digit (ISIC3 = NACE group). >=3 digits resolves;
# 2-digit codes are too coarse for the grain and go to review.
nace_long['isic3'] = np.where(nace_long['nace_level'] >= 3,
                            nace_long['nace'].str[:3], pd.NA)

# k = distinct 3-digit industries a SEGMENT touches (denominator for the 1/k split)
k_ind = (nace_long[nace_long['isic3'].notna()]
         .groupby('seg_id')['isic3'].nunique()
         .rename('k_isic3'))
nace_long = nace_long.merge(k_ind, on='seg_id', how='left')

coarse   = nace_long[nace_long['nace_level'] < 3].copy()   # 2-digit only -> review
ind_long = (nace_long[nace_long['isic3'].notna()]
            .drop_duplicates(['seg_id', 'isic3'])             # 2451 & 2452 -> one 245 row
            .copy())

print('clean 3-digit rows:', len(ind_long),
      '| distinct ISIC3:', ind_long['isic3'].nunique(),
      '| segments:', ind_long['seg_id'].nunique(),
      '| infringements:', ind_long['infringement_id'].nunique())
print('coarse (2-digit) rows routed to review:', len(coarse))

clean 3-digit rows: 169 | distinct ISIC3: 59 | segments: 149 | infringements: 146
coarse (2-digit) rows routed to review: 18


### 3.4 expand geographic scope to (infringement × country)

In [7]:
def ref_year(row):
    return row['end_year'] if (eea_membership_ref == 'infr_end'
                                 and pd.notna(row['end_year'])) else row['decision_year']

def scope_for(row):
    '''Return (set_of_iso2, scope_source) for one segment.'''
    raw = row['ctry_iso']
    yr = ref_year(row)
    if not isinstance(raw, str) or not raw.strip():
        return set(), 'missing'                 # no explicit scope -> dropped
    toks = [t.strip().upper() for t in raw.split(',') if t.strip()]
    out, src = set(), 'explicit'
    for t in toks:
        if t in ('EEA', 'EU'):
            out |= members(t, yr); src = 'bloc_expanded'
        elif t in universe:
            out.add(t)
    if not out:
        return set(), 'non_panel_only'
    return out, src

scope = infr.apply(scope_for, axis=1)
infr['ctry_set']     = scope.apply(lambda x: sorted(x[0]))
infr['scope_source'] = scope.apply(lambda x: x[1])

# carry seg_id so scope stays segment-specific (staged geographic expansion)
ctry_long = (infr[['seg_id', 'infringement_id', 'ctry_set', 'scope_source']]
             .explode('ctry_set')
             .rename(columns={'ctry_set': 'ctry_iso'})
             .dropna(subset=['ctry_iso']))

print(infr['scope_source'].value_counts(dropna=False).to_string())
print('\n(segment x country) rows:', len(ctry_long))

scope_source
bloc_expanded    103
explicit          58
missing            3

(segment x country) rows: 3079


### 3.5 build atomic treated table: (infringement × NACE4 × country)
One row per treated triple. Candidate cohorts (decision/start/end) carried side by side; **no first-treat collapse here** — that is a modelling choice made at the analysis grain in `03_panel`.

In [8]:
keep_cols = ['seg_id', 'infringement_id', 'case_id', 'cartel_name', 'case_structure',
             'segment_seq', 'decision_year', 'start_year', 'end_year',
             'has_amendment', 'k_isic3']
treated = (ind_long[keep_cols + ['isic3']]
           .merge(ctry_long[['seg_id', 'ctry_iso', 'scope_source']], on='seg_id', how='inner')
           .drop_duplicates(['seg_id', 'isic3', 'ctry_iso']))

treated = treated.rename(columns={'decision_year': 'cohort_decision',
                                  'start_year':    'cohort_start',
                                  'end_year':      'cohort_end'})
# one enforcement event = one case_id (multiple segments of a case are ONE event,
# not repeat treatment -> the 03 repeat-treatment guard keys on this)
treated['enforcement_event_id'] = treated['case_id']

print('treated rows (seg x isic3 x ctry):', len(treated))
print('segments:', treated['seg_id'].nunique(),
      '| infringements:', treated['infringement_id'].nunique(),
      '| ISIC3:', treated['isic3'].nunique(),
      '| (isic3 x ctry) cells:', treated[['isic3', 'ctry_iso']].drop_duplicates().shape[0])

treated rows (seg x isic3 x ctry): 3006
segments: 148 | infringements: 145 | ISIC3: 59 | (isic3 x ctry) cells: 1221


### 3.6 diagnostics

In [9]:
all_seg   = set(infr['seg_id'])
have_ind  = set(ind_long['seg_id'])        # >=1 clean 3-digit NACE
have_ctry = set(ctry_long['seg_id'])       # >=1 panel country
kept      = set(treated['seg_id'])

print('=== drops (intended policy: coarse NACE and missing scope are dropped) ===')
print(f'  segments in                      : {len(all_seg)}')
print(f'  dropped, coarse-only NACE (<3dig): {len(all_seg - have_ind)}')
print(f'  dropped, no panel-country scope  : {len(all_seg - have_ctry)}')
print(f'  kept in treated table            : {len(kept)}')

print('\n=== scope sources ===')
print(infr['scope_source'].value_counts().to_string())

print('\n=== treated grain ===')
print('  segments        :', treated['seg_id'].nunique())
print('  infringements   :', treated['infringement_id'].nunique())
print('  ISIC3           :', treated['isic3'].nunique())
print('  (isic3 x ctry)  :', treated[['isic3', 'ctry_iso']].drop_duplicates().shape[0])
print('  k>1 segments    :', (treated.drop_duplicates('seg_id')['k_isic3'] > 1).sum())

print('\n=== split_by_period recovery (case 40330) ===')
print(treated[treated['case_id'] == 'AT.40330']
      [['seg_id', 'segment_seq', 'isic3', 'ctry_iso', 'cohort_start', 'cohort_end']]
      .sort_values(['segment_seq', 'ctry_iso']).to_string(index=False))

=== drops (intended policy: coarse NACE and missing scope are dropped) ===
  segments in                      : 164
  dropped, coarse-only NACE (<3dig): 15
  dropped, no panel-country scope  : 3
  kept in treated table            : 148

=== scope sources ===
scope_source
bloc_expanded    103
explicit          58
missing            3

=== treated grain ===
  segments        : 148
  infringements   : 145
  ISIC3           : 59
  (isic3 x ctry)  : 1221
  k>1 segments    : 17

=== split_by_period recovery (case 40330) ===
       seg_id  segment_seq isic3 ctry_iso  cohort_start  cohort_end
AT.40330_i1_1          1.0   492       AT          2008        2014
AT.40330_i1_1          1.0   492       DE          2008        2014
AT.40330_i1_1          1.0   492       HU          2008        2014
AT.40330_i1_1          1.0   492       NL          2008        2014
AT.40330_i1_2          2.0   492       AT          2008        2014
AT.40330_i1_2          2.0   492       BE          2008        2014


### 3.7 save

In [10]:
treated.to_parquet(out_treated, index=False)

infr_out = infr.drop(columns=['nace_list', 'ctry_set'])
infr_out.to_excel(out_infr, index=False)
coarse.to_excel(out_coarse, index=False)

print(f'saved: {out_treated.name} ({len(treated):,} rows), {out_infr.name}, {out_coarse.name}')

saved: cartel_treated.parquet (3,006 rows), clean_infringements.xlsx, nace_coarse_review.xlsx


## 4. patents

### 4.1 concordance to industry key (ISIC rev4, 3-digit)

In [11]:
conc = pd.read_csv(in_ipc_isic, dtype=str)
conc.columns = ['ipc_sub', 'isic3', 'w']            # ipc4, isic_rev4_3, probability_weight
conc['w'] = conc['w'].astype(float)

wsum = conc.groupby('ipc_sub')['w'].transform('sum')
conc['w'] = conc['w'] / wsum
print(f'concordance rows: {len(conc):,} | IPC subclasses: {conc['ipc_sub'].nunique():,} '
      f'| industries: {conc['isic3'].nunique():,}')

concordance rows: 2,932 | IPC subclasses: 636 | industries: 212


### 4.2 patents to IPC subclass + fractional weight

In [12]:
pat = pd.read_parquet(in_pat)
pat = pat.dropna(subset=['appln_id', 'prio_year', 'ipc', 'ctry_code', 'app_share'])
pat['prio_year']  = pat['prio_year'].astype('int32')
pat['ipc_sub']   = pat['ipc'].astype(str).str[:4].str.strip()
pat['ctry_iso']  = pat['ctry_code'].astype(str).str.upper().str.strip()
pat['app_share'] = pd.to_numeric(pat['app_share'], errors='coerce')

# each (patent x applicant) spreads equally across its distinct IPC subclasses
pat = pat.drop_duplicates(['appln_id', 'applt_id', 'ipc_sub'])
n_sub = pat.groupby(['appln_id', 'applt_id'])['ipc_sub'].transform('count')
pat['frac_weight'] = (pat['app_share'] / n_sub).astype('float64')
pat['fw_cit3'] = pat['frac_weight'] * pd.to_numeric(pat['cit_fwd_3yr'], errors='coerce').fillna(0)
pat['fw_cit']  = pat['frac_weight'] * pd.to_numeric(pat['cit_fwd'],     errors='coerce').fillna(0)

print(f'patent rows: {len(pat):,} | applications: {pat['appln_id'].nunique():,}')

patent rows: 13,798,349 | applications: 6,896,974


### 4.3 assign to industry

In [13]:
matched = pat['ipc_sub'].isin(set(conc['ipc_sub']))
print(f'IPC-subclass rows matched to concordance: {matched.mean():.1%} '
      f'({(~matched).sum():,} unmatched rows dropped)')

# collapse to (ctry, year, ipc_sub) then expand to industry via concordance weights
ipc_cell = (pat.groupby(['ctry_iso', 'prio_year', 'ipc_sub'], observed=True)
              .agg(frac_weight=('frac_weight', 'sum'),
                   fw_cit3    =('fw_cit3', 'sum'),
                   fw_cit     =('fw_cit', 'sum'))
              .reset_index())
ipc_cell = ipc_cell.merge(conc, on='ipc_sub', how='inner')

for src, dst in [('frac_weight', 'pat_frac'), ('fw_cit3', 'pat_cit3_frac'), ('fw_cit', 'pat_cit_frac')]:
    ipc_cell[dst] = (ipc_cell[src] * ipc_cell['w']).astype('float32')

panel = (ipc_cell.groupby(['ctry_iso', 'isic3', 'prio_year'], observed=True)
         .agg(pat_frac     =('pat_frac', 'sum'),
              pat_cit3_frac=('pat_cit3_frac', 'sum'),
              pat_cit_frac =('pat_cit_frac', 'sum'))
         .reset_index().rename(columns={'prio_year': 'year'}))
del ipc_cell

# whole counts
ipc_to_ind = conc[['ipc_sub', 'isic3']].drop_duplicates()
cnt_rows = pat[['appln_id', 'applt_id', 'ctry_iso', 'prio_year', 'ipc_sub']].merge(
    ipc_to_ind, on='ipc_sub', how='inner')

counts = (cnt_rows.groupby(['ctry_iso', 'isic3', 'prio_year'], observed=True)
          .agg(n_pat_appln=('appln_id', 'nunique'),
               n_applt    =('applt_id', 'nunique'))
          .reset_index().rename(columns={'prio_year': 'year'}))

panel = panel.merge(counts, on=['ctry_iso', 'isic3', 'year'], how='left')
panel[['n_pat_appln', 'n_applt']] = panel[['n_pat_appln', 'n_applt']].fillna(0).astype(int)
del cnt_rows, counts

IPC-subclass rows matched to concordance: 98.7% (182,865 unmatched rows dropped)


### 4.4 restrict to country scope

In [14]:
n0 = len(panel)
panel = panel[panel['ctry_iso'].isin(europe_iso2)].copy()
print(f'country filter to EU/EEA universe: {n0:,} -> {len(panel):,} cells')
print(f'panel: {len(panel):,} cell-years | {panel['isic3'].nunique()} industries | '
      f'{panel['ctry_iso'].nunique()} countries | {panel['year'].min()}-{panel['year'].max()}')

country filter to EU/EEA universe: 391,276 -> 202,241 cells
panel: 202,241 cell-years | 212 industries | 49 countries | 1961-2024


### 4.5 save

In [15]:
panel.to_parquet(out_patpanel, index=False)
print(f'saved: {out_patpanel.name} ({len(panel):,} rows)')

del pat, panel; gc.collect()

saved: pat_panel.parquet (202,241 rows)


7854